# Module P05 — Exécution nominale d'un mouvement

**Objectif :** coder `executer_mouvement()` pour le chemin sans erreur :  
déplacer un drone, consommer la batterie, gérer prise et livraison de survivant.

> Compatible Google Colab — aucun fichier externe requis.

## Section 1 — Rappels synthétiques

### Ce qu'on a vu dans P04 (prérequis)
| Fonction | Rôle | Retour |
|----------|------|--------|
| `valider_mouvement()` | Vérifie si le mouvement est légal | `(bool, str)` |
| `executer_mouvement()` | Applique le mouvement (ce module) | `str` (log) |

### Règle clé : double mise à jour obligatoire
Quand un drone se déplace, **deux** structures doivent être synchronisées :
1. `etat['drones'][id_drone]['col']` et `'lig'`
2. `etat['grille'][lig][col]` — effacer l'ancienne case, marquer la nouvelle

### Coût batterie (cas nominal)
| Situation | Coût |
|-----------|------|
| Déplacement seul | -1 |
| Déplacement + transport survivant | -2 |
| Stationnaire ou arrivée hôpital | +3 (recharge) |
| Livraison d'un survivant à l'hôpital | score +1 |


In [ ]:
# === Configuration initiale (reproduite inline pour Colab) ===
GRILLE_TAILLE = 12
BATTERIE_MAX = 20
COLS = 'ABCDEFGHIJKL'

def col_vers_index(lettre):
    return COLS.index(lettre.upper())

def index_vers_col(i):
    return COLS[i]

In [ ]:
# === État minimal pour tester P05 ===
grille = [['.' for _ in range(12)] for _ in range(12)]

etat = {
    'tour': 1,
    'score': 0,
    'partie_finie': False,
    'victoire': False,
    'grille': grille,
    'hopital': (0, 0),          # colonne A, ligne 1
    'batiments': [],
    'drones': {
        'D1': {
            'id': 'D1', 'col': 3, 'lig': 3,
            'batterie': 15, 'batterie_max': 20,
            'survivant': None, 'bloque': 0, 'hors_service': False
        }
    },
    'tempetes': {},
    'survivants': {
        'S1': {'id': 'S1', 'col': 4, 'lig': 3, 'etat': 'en_attente'}
    },
    'zones_x': set(),
    'historique': []
}

# Placer les entités sur la grille
etat['grille'][0][0] = 'H'      # hôpital en (col=0, lig=0)
etat['grille'][3][3] = 'D1'     # drone D1 en (col=3, lig=3)
etat['grille'][3][4] = 'S1'     # survivant S1 en (col=4, lig=3)

print('Grille initialisée ✓')
print(f"D1 position initiale : col={etat['drones']['D1']['col']}, lig={etat['drones']['D1']['lig']}")
print(f"S1 état : {etat['survivants']['S1']['etat']}")

## Section 2 — Exercice guidé

### Étape 1 — Déplacement simple (sans survivant)
Commence par le cas le plus simple : déplacer D1 d'une case vers (col=4, lig=3).

In [ ]:
def executer_mouvement_v1(etat, id_drone, col_cible, lig_cible):
    """Version 1 — déplacement simple, sans gestion survivant ni hôpital."""
    drone = etat['drones'][id_drone]

    # TODO 1 : sauvegarder col et lig avant déplacement
    col_avant = None  # à compléter
    lig_avant = None  # à compléter

    # TODO 2 : effacer l'ancienne position sur la grille
    # etat['grille'][...][...] = '.'

    # TODO 3 : mettre à jour col et lig du drone
    # drone['col'] = ...
    # drone['lig'] = ...

    # TODO 4 : consommer la batterie (-1)
    # drone['batterie'] -= ...

    # TODO 5 : marquer la nouvelle position sur la grille
    # etat['grille'][...][...] = id_drone

    return f"{id_drone} déplacé en ({col_cible},{lig_cible})"

In [ ]:
# Vérification étape 1
# Remettre l'état initial
etat['drones']['D1']['col'] = 3
etat['drones']['D1']['lig'] = 3
etat['drones']['D1']['batterie'] = 15
etat['grille'][3][3] = 'D1'
etat['grille'][3][4] = 'S1'

# Test : déplacer D1 vers (col=3, lig=4) — une case en bas
# Note : on évite (4,3) qui contient S1, pour tester sans survivant
log = executer_mouvement_v1(etat, 'D1', 3, 4)
print('Log :', log)

assert etat['drones']['D1']['col'] == 3, 'col incorrecte'
assert etat['drones']['D1']['lig'] == 4, 'lig incorrecte'
assert etat['drones']['D1']['batterie'] == 14, 'batterie incorrecte'
assert etat['grille'][3][3] == '.', 'ancienne case non effacée'
assert etat['grille'][4][3] == 'D1', 'nouvelle case non marquée'
print('✓ Étape 1 validée')

### Étape 2 — Prise d'un survivant
Quand le drone arrive sur la case d'un survivant `en_attente`, il l'embarque :
- `drone['survivant'] = id_survivant`
- `etat['survivants'][id_survivant]['etat'] = 'embarque'`

In [ ]:
def executer_mouvement_v2(etat, id_drone, col_cible, lig_cible):
    """Version 2 — déplacement + prise de survivant."""
    drone = etat['drones'][id_drone]
    col_avant, lig_avant = drone['col'], drone['lig']

    etat['grille'][lig_avant][col_avant] = '.'
    drone['col'], drone['lig'] = col_cible, lig_cible

    # Coût batterie
    cout = 1
    if drone['survivant'] is not None:
        cout += 1
    drone['batterie'] -= cout

    # TODO : parcourir etat['survivants'] et vérifier si un survivant
    # 'en_attente' se trouve sur (col_cible, lig_cible).
    # Si oui : drone['survivant'] = sid, surv['etat'] = 'embarque'
    # for sid, surv in etat['survivants'].items():
    #     if ...:
    #         ...
    #         break

    etat['grille'][lig_cible][col_cible] = id_drone
    return f"{id_drone} en ({col_cible},{lig_cible}), batterie={drone['batterie']}"

In [ ]:
# Vérification étape 2 — D1 se déplace sur S1
etat['drones']['D1']['col'] = 3
etat['drones']['D1']['lig'] = 3
etat['drones']['D1']['batterie'] = 15
etat['drones']['D1']['survivant'] = None
etat['survivants']['S1']['etat'] = 'en_attente'
etat['grille'][3][3] = 'D1'
etat['grille'][3][4] = 'S1'

log = executer_mouvement_v2(etat, 'D1', 4, 3)  # D1 → (col=4, lig=3) où est S1
print('Log :', log)

assert etat['drones']['D1']['survivant'] == 'S1', 'D1 doit embarquer S1'
assert etat['survivants']['S1']['etat'] == 'embarque', 'S1 doit être embarqué'
assert etat['drones']['D1']['batterie'] == 14, 'batterie incorrecte (1 déplacement seul)'
print('✓ Étape 2 validée')

### Étape 3 — Livraison à l'hôpital
Quand le drone arrive à l'hôpital avec un survivant :
- `etat['survivants'][sid]['etat'] = 'sauve'`
- `drone['survivant'] = None`
- `etat['score'] += 1`
- Recharge batterie : `min(batterie + 3, batterie_max)`

In [ ]:
def executer_mouvement_v3(etat, id_drone, col_cible, lig_cible):
    """Version 3 — déplacement + prise + livraison + recharge."""
    drone = etat['drones'][id_drone]
    col_avant, lig_avant = drone['col'], drone['lig']

    etat['grille'][lig_avant][col_avant] = '.'
    drone['col'], drone['lig'] = col_cible, lig_cible

    cout = 1
    if drone['survivant'] is not None:
        cout += 1
    drone['batterie'] -= cout

    # Prise survivant
    for sid, surv in etat['survivants'].items():
        if surv['col'] == col_cible and surv['lig'] == lig_cible \
                and surv['etat'] == 'en_attente':
            drone['survivant'] = sid
            surv['etat'] = 'embarque'
            break

    # TODO : gérer livraison à l'hôpital
    # Si (col_cible, lig_cible) == etat['hopital'] ET drone['survivant'] is not None :
    #   - passer le survivant en 'sauve'
    #   - drone['survivant'] = None
    #   - etat['score'] += 1
    #   - recharger : drone['batterie'] = min(drone['batterie'] + 3, drone['batterie_max'])
    hop_col, hop_lig = etat['hopital']
    # if ...:
    #     ...

    etat['grille'][lig_cible][col_cible] = id_drone
    return f"{id_drone} en ({col_cible},{lig_cible}), score={etat['score']}, batterie={drone['batterie']}"

In [ ]:
# Vérification étape 3 — D1 avec S1 embarqué va à l'hôpital (0,0)
etat['drones']['D1']['col'] = 1
etat['drones']['D1']['lig'] = 0
etat['drones']['D1']['batterie'] = 10
etat['drones']['D1']['survivant'] = 'S1'
etat['survivants']['S1']['etat'] = 'embarque'
etat['score'] = 0
etat['grille'][0][1] = 'D1'
etat['grille'][0][0] = 'H'

log = executer_mouvement_v3(etat, 'D1', 0, 0)  # D1 → hôpital (col=0, lig=0)
print('Log :', log)

assert etat['score'] == 1, 'score doit être 1'
assert etat['drones']['D1']['survivant'] is None, 'drone doit avoir libéré le survivant'
assert etat['survivants']['S1']['etat'] == 'sauve', 'S1 doit être sauvé'
assert etat['drones']['D1']['batterie'] == 11, 'batterie : 10 - 2 (transport) + 3 (hôpital) = 11'
print('✓ Étape 3 validée — module P05 complet !')